# 2-3. 소분류별 날씨 민감도 분석

목적: 전체를 뭉뚱그려서 보면 신호가 희석되므로, 소분류(80여개) 단위로 날씨(기온/강수)와의
상관관계를 직접 계산해 "날씨에 명확히 반응하는 카테고리"를 정량적으로 식별한다.

분석 단위: (날짜, 권역, 소분류) — 같은 날짜라도 권역별 날씨가 다르므로, 시간적 변화와
지역적 차이를 동시에 활용해 표본을 확보한다.

'불필요' 그룹은 대조군으로 포함한다 — 날씨와 무관하다고 판단했던 카테고리들의 상관계수가
실제로 낮게 나오는지 검증하는 용도다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

system = platform.system()
if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2-3-0. 데이터 로드 및 집계

In [ ]:
from pathlib import Path

CATEGORY_DIR = Path("../../data/processed/consume_weather_by_category")
GROUP_ORDER = ["필수", "애매", "불필요"]

group_dfs = {}
for group in GROUP_ORDER:
    g_df = pd.read_parquet(CATEGORY_DIR / f"{group}.parquet")
    g_df["분류등급"] = group
    group_dfs[group] = g_df

df = pd.concat(group_dfs.values(), ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

# (날짜, 권역, 소분류) 단위로 집계 -- 도시/연령그룹은 합산해서 흡수
agg = (
    df.groupby(["date", "지점명", "소분류", "분류등급"])
    .agg(매출금액=("매출금액", "sum"), 평균기온=("평균기온(°C)", "first"), 강수량=("일강수량(mm)", "first"))
    .reset_index()
)
agg["log_매출금액"] = np.log1p(agg["매출금액"])

print("집계 후 shape:", agg.shape)
agg.head()

## 2-3-1. 소분류별 기온/강수 상관계수 계산

스피어만 상관계수(순위 기반, 비선형 관계와 이상치에 강함)를 사용한다.
소분류당 표본이 너무 적으면(예: 20개 미만) 상관계수가 불안정하므로 제외한다.

In [ ]:
results = []
for (category, group), sub in agg.groupby(["소분류", "분류등급"]):
    if len(sub) < 20:
        continue

    temp_corr, temp_p = stats.spearmanr(sub["평균기온"], sub["log_매출금액"])
    rain_corr, rain_p = stats.spearmanr(sub["강수량"], sub["log_매출금액"])

    results.append({
        "소분류": category,
        "분류등급": group,
        "표본수": len(sub),
        "기온상관": temp_corr,
        "기온_p값": temp_p,
        "강수상관": rain_corr,
        "강수_p값": rain_p,
    })

sensitivity_df = pd.DataFrame(results).sort_values("기온상관", ascending=False)
print(f"총 {len(sensitivity_df)}개 소분류 분석됨")
sensitivity_df

## 2-3-2. 날씨 민감도 4분류

- 고온민감: 기온상관 > 0.3 (유의미하게 더울수록 매출 증가)
- 저온민감: 기온상관 < -0.3 (유의미하게 추울수록 매출 증가)
- 강수민감: |강수상관| > 0.15 (기온 기준을 만족 안 해도, 비 여부에 따라 뚜렷한 차이)
- 날씨무관: 위 어디에도 해당 안 함

임계값은 잠정 기준이며, 실제 분포를 보고 조정 가능하다.

In [ ]:
def classify(row):
    if row["기온상관"] > 0.3:
        return "고온민감"
    elif row["기온상관"] < -0.3:
        return "저온민감"
    elif abs(row["강수상관"]) > 0.15:
        return "강수민감"
    else:
        return "날씨무관"

sensitivity_df["날씨민감도"] = sensitivity_df.apply(classify, axis=1)

print(sensitivity_df["날씨민감도"].value_counts())
print()
print("=== 분류등급 x 날씨민감도 교차표 (검증: 불필요 그룹은 '날씨무관'이 많아야 함) ===")
print(pd.crosstab(sensitivity_df["분류등급"], sensitivity_df["날씨민감도"]))

## 2-3-3. 고온민감 / 저온민감 상위 카테고리

In [ ]:
top_hot = sensitivity_df.nlargest(10, "기온상관")[["소분류", "분류등급", "기온상관", "기온_p값", "표본수"]]
top_cold = sensitivity_df.nsmallest(10, "기온상관")[["소분류", "분류등급", "기온상관", "기온_p값", "표본수"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_hot["소분류"][::-1], top_hot["기온상관"][::-1], color="tomato")
axes[0].set_title("고온민감 상위 10개 소분류 (기온상관 높은 순)")
axes[0].set_xlabel("기온상관계수")

axes[1].barh(top_cold["소분류"][::-1], top_cold["기온상관"][::-1], color="steelblue")
axes[1].set_title("저온민감 상위 10개 소분류 (기온상관 낮은 순)")
axes[1].set_xlabel("기온상관계수")

plt.tight_layout()
plt.show()

print(top_hot)
print()
print(top_cold)

## 2-3-4. 강수민감 상위 카테고리

In [ ]:
top_rain = sensitivity_df.reindex(sensitivity_df["강수상관"].abs().sort_values(ascending=False).index).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["seagreen" if v > 0 else "gray" for v in top_rain["강수상관"]]
ax.barh(top_rain["소분류"][::-1], top_rain["강수상관"][::-1], color=colors[::-1])
ax.set_title("강수민감 상위 10개 소분류 (|강수상관| 높은 순)")
ax.set_xlabel("강수상관계수")
plt.tight_layout()
plt.show()

top_rain[["소분류", "분류등급", "강수상관", "강수_p값", "표본수"]]

## 2-3-5. 전체 지형도 (기온상관 x 강수상관 산점도)

분류등급별로 색을 다르게 해서, '불필요' 그룹이 실제로 원점 근처(날씨 무관)에
몰려있는지 시각적으로 확인한다.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors = {"필수": "tomato", "애매": "goldenrod", "불필요": "gray"}

for g in GROUP_ORDER:
    sub = sensitivity_df[sensitivity_df["분류등급"] == g]
    ax.scatter(sub["기온상관"], sub["강수상관"], label=g, alpha=0.7, color=colors[g])

ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("기온상관계수")
ax.set_ylabel("강수상관계수")
ax.set_title("소분류별 날씨 민감도 지형도 (분류등급별 색상)")
ax.legend()
plt.tight_layout()
plt.show()

## 2-3 요약 (직접 채워넣기)

- 고온민감 카테고리: ...
- 저온민감 카테고리: ...
- 강수민감 카테고리: ...
- '불필요' 그룹 대조군 검증 결과: [예상대로 날씨무관이 많았음 / 예상과 다름] ...

다음 단계: 식별된 날씨 반응형 카테고리만 추려서 회귀분석/심화 모델링 진행.